# EDA Basis

In [ ]:
# General TODOs:

# General TODOs:

# TODO: ...

# General NOTEs:

# NOTE: ...

# Setup

## Imports

In [ ]:
import warnings
from typing import Any

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
from statsmodels.tools.sm_exceptions import InterpolationWarning
from statsmodels.tsa.stattools import adfuller, kpss

from thesis_project import (
    config,
)

## Paths

In [ ]:
TICKER = "fbtp"
NOTEBOOK = "eda-basis"

# Lobs directories
LOB_DIR = config.RAW_DIR / TICKER
LOB_FUT_DIR = LOB_DIR / "futures"
LOB_CTD_DIR = LOB_DIR / "ctd"

# Output directories
if NOTEBOOK is not None:
    OUT_FIG_DIR = config.FIG_DIR / TICKER / NOTEBOOK
    OUT_FIG_DIR.mkdir(parents=True, exist_ok=True)
    OUT_TAB_DIR = config.TAB_DIR / TICKER / NOTEBOOK
    OUT_TAB_DIR.mkdir(parents=True, exist_ok=True)
    # OUT_INT_DIR = config.INT_DIR / TICKER / NOTEBOOK
    # OUT_INT_DIR.mkdir(parents=True, exist_ok=True)
    # OUT_PRO_DIR = config.PRO_DIR / TICKER / NOTEBOOK
    # OUT_PRO_DIR.mkdir(parents=True, exist_ok=True)

## Settings

In [ ]:
# Matplotlib global settings
# config.MPL contains a personalized default configuration for matplotlib
plt.rcdefaults()
plt.style.use("seaborn-v0_8-paper")

In [ ]:
# Warnings settings
warnings.simplefilter("always", UserWarning)

In [ ]:
# Sample date for quick analysis
year = "2022"
month = "09"
day = "07"
date_str = f"{year}_{month}_{day}"
fut_filename = f"futures_lob_freq_1s_{date_str}.parquet"
ctd_filename = f"ctd_lob_freq_1s_{date_str}.parquet"

## Utilities

### Plotting

# Main

## Common

In [ ]:
# Basis data
filename = "gross_basis_freq_1s_from_2022_08_01_to_2024_12_30.parquet"
basis_df = pd.read_parquet(config.INT_DIR / TICKER / "data-microstructure" / "basis" / filename)
assert isinstance(basis_df.index, pd.DatetimeIndex)

tz = basis_df.index.tz
idx = basis_df.index

# Compute basis 'returns' (first difference)
basis_diff_freqs = {
    "1s": 1,  # seconds
    "5s": 5,
    "10s": 10,
    "30s": 30,
    "1min": 60,
    "5min": 300,
    "10min": 600,
    "30min": 1800,
    "1h": 3600,
}

for key, freq in basis_diff_freqs.items():
    basis_df[f"Basis diff {key}"] = (
        basis_df["Basis"]
        .groupby(
            basis_df.index.normalize()
        )  # Group by day to avoid computing overnight differences
        .diff(freq)
    )

In [ ]:
fut_meta_df = pd.read_csv(
    LOB_DIR / "fut_metadata.csv", parse_dates=["Last Trading Date", "Delivery Date"]
)

ctd_switch_df = pd.read_csv(LOB_DIR / "ctd_switch.csv", parse_dates=["CTD Switch Date"])

In [ ]:
# Get filtered relevant dates
start = basis_df.index.min().normalize()
end = basis_df.index.max().normalize()

# Last trading dates of futures
last_trading_dates = pd.to_datetime(fut_meta_df["Last Trading Date"].values)
last_trading_dates = last_trading_dates.tz_localize(tz).normalize()
filtered_last_trading_dates = last_trading_dates[
    (last_trading_dates >= start) & (last_trading_dates <= end)
]

# CTD switch dates
ctd_switch_dates = pd.to_datetime(ctd_switch_df["CTD Switch Date"].values)
ctd_switch_dates = ctd_switch_dates.tz_localize(tz).normalize()
filtered_ctd_switch_dates = ctd_switch_dates[
    (ctd_switch_dates >= start) & (ctd_switch_dates <= end)
]

## Distribution

In [ ]:
display(basis_df.describe())

In [ ]:
# Plot basis timeseries
# NOTE: Not sure if it is okay to slightly cut the tails of the distribution
# NOTE: Not sure if it is better to use density=True or weights, weights are easier to interpret while density give us an actual pdf
import matplotlib.ticker as mtick
import numpy as np

start_date = basis_df.index.min()
end_date = basis_df.index.max()

plot_data = basis_df["Basis"]
upper = plot_data.quantile(0.999)
lower = plot_data.quantile(0.001)
mask = plot_data.between(lower, upper)
plot_data = plot_data[mask]
weights = np.ones_like(plot_data) / len(plot_data)

fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(plot_data, bins=600, weights=weights)
title = (
    "Basis Distribution"
    f"\nFrom {start_date.strftime('%Y-%m-%d')} To {end_date.strftime('%Y-%m-%d')}"
    f"\nQuantiles 0.1% - 99.9%"
)
ax.set_title(title)
ax.set_xlabel("Basis")
ax.set_ylabel("Frequency")
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
fig.tight_layout()
plt.show()

In [ ]:
# Plot basis 1st difference timeseries
import matplotlib.ticker as mtick
import numpy as np

start_date = basis_df.index.min()
end_date = basis_df.index.max()

for key, freq in basis_diff_freqs.items():
    plot_data = basis_df[f"Basis diff {key}"]
    upper = plot_data.quantile(0.999)
    lower = plot_data.quantile(0.001)
    mask = plot_data.between(lower, upper)
    plot_data = plot_data[mask]
    weights = np.ones_like(plot_data) / len(plot_data)

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.hist(
        plot_data,
        bins=750,
        weights=weights,
        # density=True
    )
    title = (
        f"Basis 1st Difference {key} Distribution"
        f"\nFrom {start_date.strftime('%Y-%m-%d')} To {end_date.strftime('%Y-%m-%d')}"
        f"\nQuantiles 0.1% - 99.9%"
    )
    ax.set_title(title)
    ax.set_xlabel("Basis 1st Difference")
    ax.set_ylabel("Frequency")
    ax.grid(True, alpha=0.3)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    fig.tight_layout()
    plt.show()

## Time Series

### Whole Time Series

In [ ]:
# Plot basis timeseries
start_date = basis_df.index.min()
end_date = basis_df.index.max()
start_xlim = start_date - pd.Timedelta(days=35)
end_xlim = end_date + pd.Timedelta(days=15)

resample_freq = "1min"
plot_data = (
    basis_df.resample(resample_freq)["Basis"]  # Resample to reduce number of poits to plot
    .last()
    .dropna()
)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(
    plot_data,
    label="Basis",
    linewidth=0.5,
)
title = (
    "Basis Time Series\n"
    f"From {start_date.strftime('%Y-%m-%d')} To {end_date.strftime('%Y-%m-%d')}\n"
    f"Resampled at {resample_freq} frequency (mean)"
)
ax.set_title(title)
ax.set_xlabel("Time")
ax.set_ylabel("Basis")
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_locator(mdates.MonthLocator(range(1, 13), interval=3, tz=tz))
ax.set_xlim(start_xlim, end_xlim)
for d in filtered_last_trading_dates:
    if d == filtered_last_trading_dates.min():
        ax.axvline(d, color="red", linestyle="--", linewidth=0.5, label="Last Trading Date")
    else:
        ax.axvline(
            d,
            color="red",
            linestyle="--",
            linewidth=0.5,
        )
# for d in filtered_ctd_switch_dates:
#     if d == filtered_ctd_switch_dates.min():
#         ax.axvline(
#             d,
#             color='blue',
#             linestyle='--',
#             linewidth=0.5,
#             label='CTD Switch Date'
#         )
#     else:
#         ax.axvline(
#             d,
#             color='blue',
#             linestyle='--',
#             linewidth=0.5,
#         )
ax.legend(loc="upper left")
fig.tight_layout()
plt.show()

In [ ]:
# Plot diff timeseries
start_date = basis_df.index.min()
end_date = basis_df.index.max()
start_xlim = start_date - pd.Timedelta(days=35)
end_xlim = end_date + pd.Timedelta(days=15)
resample_freq = "1min"

for key, freq in basis_diff_freqs.items():
    plot_data = basis_df.resample(resample_freq)[f"Basis diff {key}"].mean().dropna()

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(
        plot_data,
        label=f"Basis 1st difference {key}",
        linewidth=0.5,
    )
    title = (
        f"Basis 1st Difference {key} Time Series\n"
        f"From {start_date.strftime('%Y-%m-%d')} To {end_date.strftime('%Y-%m-%d')}\n"
        f"Resampled at {resample_freq} frequency (mean)"
    )
    ax.set_title(title)
    ax.set_xlabel("Time")
    ax.set_ylabel("Basis 1st Difference")
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_locator(mdates.MonthLocator(range(1, 13), interval=3, tz=tz))
    ax.set_xlim(start_xlim, end_xlim)
    for d in filtered_last_trading_dates:
        if d == filtered_last_trading_dates.min():
            ax.axvline(d, color="red", linestyle="--", linewidth=0.5, label="Last Trading Date")
        else:
            ax.axvline(
                d,
                color="red",
                linestyle="--",
                linewidth=0.5,
            )
    # for d in filtered_ctd_switch_dates:
    #     if d == filtered_ctd_switch_dates.min():
    #         ax.axvline(
    #             d,
    #             color='blue',
    #             linestyle='--',
    #             linewidth=0.5,
    #             label='CTD Switch Date'
    #         )
    #     else:
    #         ax.axvline(
    #             d,
    #             color='blue',
    #             linestyle='--',
    #             linewidth=0.5,
    #         )
    ax.legend(loc="upper left")
    fig.tight_layout()
    plt.show()

### Day Sample

In [ ]:
# Plot a sample day of basis and basis first differences
assert isinstance(basis_df.index, pd.DatetimeIndex)

date = pd.Timestamp(f"{year}-{month}-{day}", tz=basis_df.index.tz)
date_mask = basis_df.index.normalize() == date
start_xlim = date + pd.Timedelta(hours=7, minutes=30)
end_xlim = date + pd.Timedelta(hours=18, minutes=30)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(
    basis_df.loc[date_mask, "Basis"],
    label="Basis",
    linewidth=0.5,
)
title = f"Basis Time Series on {date_str.replace('_', '-')}"
ax.set_title(title)
ax.set_xlabel("Time")
ax.set_ylabel("Basis")
ax.grid(True, alpha=0.3)
ax.legend()
ax.xaxis.set_major_locator(mdates.HourLocator(interval=1, tz=basis_df.index.tz))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M", tz=basis_df.index.tz))
ax.set_xlim(start_xlim, end_xlim)
fig.tight_layout()
plt.show()

In [ ]:
assert isinstance(basis_df.index, pd.DatetimeIndex)

for key, freq in basis_diff_freqs.items():
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(
        basis_df.loc[date_mask, f"Basis diff {key}"],
        label=f"Basis diff {key}",
        linewidth=0.5,
    )
    title = f"Basis 1st Difference {key} Time Series on {date_str.replace('_', '-')}"
    ax.set_title(title)
    ax.set_xlabel("Time")
    ax.set_ylabel("Basis Diff")
    ax.grid(True, alpha=0.3)
    ax.legend()
    ax.xaxis.set_major_locator(mdates.HourLocator(interval=1, tz=basis_df.index.tz))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M", tz=basis_df.index.tz))
    ax.set_xlim(start_xlim, end_xlim)
    fig.tight_layout()
    plt.show()

## Stationarity Tests

ADF:

$\Delta y_t = \alpha + \beta t + \gamma y_{t-1} + \sum_{i=1}^p \Delta y_{t-i} + \varepsilon_t$

$\begin{cases} H_0: \text{The series is not stationary} \\ H_1: \text{The series is stationary} \end{cases}, \text{p-value} \le 0.05 \implies \text{Reject } H_0 \implies \text{Stationarity}$

KPSS:

$y_t = \mu_t + \varepsilon_t$

$\begin{cases} H_0: \text{The series is stationary} \\ H_1: \text{The series is not stationary} \end{cases}, \text{p-value} \le 0.05 \implies \text{Reject } H_0 \implies \text{No stationarity}$

From *statsmodels* documentation:

1. Both tests conclude that the series is not stationary - The series is not stationary

2. Both tests conclude that the series is stationary - The series is stationary

3. KPSS indicates stationarity and ADF indicates non-stationarity - The series is trend stationary. Trend needs to be removed to make series strict stationary. The detrended series is checked for stationarity.

4. KPSS indicates non-stationarity and ADF indicates stationarity - The series is difference stationary. Differencing is to be used to make series stationary. The differenced series is checked for stationarity.


### Day-wise Basis Stationarity Analysis

For each day in the LOBs dataset we examine the stationarity of the basis.

In [ ]:
# Perform day-wise stationarity tests
assert isinstance(basis_df.index, pd.DatetimeIndex)
warnings.filterwarnings("ignore", category=InterpolationWarning)

resample_freqs = [
    "1s",
    "5s",
    "10s",
    "15s",
    "30s",
    "1min",
]

results = []
for date, group in list(basis_df.groupby(basis_df.index.normalize())):
    row: dict[str, Any] = {"Date": date.date()}
    for freq in resample_freqs:
        adf_stat, adf_pvalue, *_ = adfuller((group["Basis"].resample(freq).last().dropna()))
        kpss_stat, kpss_pvalue, *_ = kpss((group["Basis"].resample(freq).last().dropna()))
        row[f"ADF Statistic {freq}"] = adf_stat
        row[f"ADF p-value {freq}"] = adf_pvalue
        row[f"KPSS Statistic {freq}"] = kpss_stat
        row[f"KPSS p-value {freq}"] = kpss_pvalue
    results.append(row)

results_df = pd.DataFrame(results)
results_df.set_index("Date", inplace=True)

In [ ]:
path = config.INT_DIR / TICKER / "data-microstructure" / "basis"
filename = "gross_basis_stationarity_tests.parquet"
# results_df.to_parquet(path / filename)
results_df = pd.read_parquet(path / filename)
display(results_df.head())

In [ ]:
bins = np.linspace(0, 1, 101)
for freq in resample_freqs:
    print(f"ADF stationary ratio: {(results_df[f'ADF p-value {freq}'] <= 0.05).mean() * 100:.2f} %")
    print(
        f"KPSS stationary ratio: {(results_df[f'KPSS p-value {freq}'] > 0.05).mean() * 100:.2f} %"
    )
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(
        results_df[f"ADF p-value {freq}"],
        bins=bins,  # type: ignore
        alpha=0.5,
        label="ADF",
    )
    ax.hist(
        results_df[f"KPSS p-value {freq}"],
        bins=bins,  # type: ignore
        alpha=0.5,
        label="KPSS",
    )
    title = (
        f"Distribution of ADF and KPSS p-values for Basis Resampled at {freq} Frequency"
        f"\nFrom {results_df.index.min()} To {results_df.index.max()}"
    )
    if freq != "1s":
        title += f"\nResampled at {freq} frequency (last)"
    ax.set_title(title)
    ax.set_xlabel("p-value")
    ax.set_ylabel("Frequency")
    ax.set_xticks(bins[:21:5])
    ax.set_xlim(0, 0.2)
    ax.grid(True, alpha=0.3)
    ax.axvline(0.05, color="red", linestyle="-", linewidth=0.5)
    ax.legend()
    plt.tight_layout()
    plt.show()

### Day-wise Basis 1st Difference Stationarity Analysis

For each day in the LOBs dataset we examine the stationarity of the basis 1st difference for different frequencies.

In [ ]:
# Perform day-wise stationarity tests on 1st differences
assert isinstance(basis_df.index, pd.DatetimeIndex)
warnings.filterwarnings("ignore", category=InterpolationWarning)

results = []
for date, group in list(basis_df.groupby(basis_df.index.normalize())):
    row: dict[str, Any] = {"Date": date.date()}
    for key, freq in basis_diff_freqs.items():
        adf_stat, adf_pvalue, *_ = adfuller((group[f"Basis diff {key}"].dropna()))
        kpss_stat, kpss_pvalue, *_ = kpss((group[f"Basis diff {key}"].dropna()))
        row[f"ADF Statistic Difference {key}"] = adf_stat
        row[f"ADF p-value Difference {key}"] = adf_pvalue
        row[f"KPSS Statistic Difference {key}"] = kpss_stat
        row[f"KPSS p-value Difference {key}"] = kpss_pvalue
    results.append(row)

results_df = pd.DataFrame(results)
results_df.set_index("Date", inplace=True)

In [ ]:
path = config.INT_DIR / TICKER / "data-microstructure" / "basis"
filename = "gross_basis_diff_stationarity_tests.parquet"
# results_df.to_parquet(path / filename)
results_df = pd.read_parquet(path / filename)
display(results_df.head())

In [ ]:
bins = np.linspace(0, 1, 101)
for key, _ in basis_diff_freqs.items():
    print(
        f"ADF stationary ratio: {(results_df[f'ADF p-value Difference {key}'] <= 0.05).mean() * 100:.2f} %"
    )
    print(
        f"KPSS stationary ratio: {(results_df[f'KPSS p-value Difference {key}'] > 0.05).mean() * 100:.2f} %"
    )
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(
        results_df[f"ADF p-value Difference {key}"],
        bins=bins,  # type: ignore
        alpha=0.5,
        label="ADF",
    )
    ax.hist(
        results_df[f"KPSS p-value Difference {key}"],
        bins=bins,  # type: ignore
        alpha=0.5,
        label="KPSS",
    )
    title = (
        f"Distribution of ADF and KPSS p-values for Basis 1st Difference {key}"
        f"\nFrom {results_df.index.min()} To {results_df.index.max()}"
    )
    ax.set_title(title)
    ax.set_xlabel("p-value")
    ax.set_ylabel("Frequency")
    ax.set_xticks(bins[:21:5])
    ax.set_xlim(0, 0.2)
    ax.grid(True, alpha=0.3)
    ax.axvline(0.05, color="red", linestyle="-", linewidth=0.5)
    ax.legend()
    plt.tight_layout()
    plt.show()

### Across-day Basis Stationarity Analysis

For each valid second of the day we examine the stationarity of the basis across days.

In [ ]:
# TODO: Implement the stationarity tests

## Boxplots

In [ ]:
import os
from typing import Optional


def plot_grouped_boxplot(
    df: pd.DataFrame,
    value_col: str,
    group_col: str,
    order: list,
    tick_labels: list,
    showfliers: bool = False,
    title: Optional[str] = None,
    ylabel: Optional[str] = None,
    xlabel: Optional[str] = None,
    tick_positions: Optional[list] = None,
    save_path: Optional[str] = None,
):
    plot_data = [df[df[group_col] == val][value_col].dropna().values for val in order]

    fig, ax = plt.subplots(figsize=(10, 6))

    # tick_labels is only passed if tick_positions is None, otherwise we set it explicitly later
    ax.boxplot(
        plot_data,
        tick_labels=tick_labels if tick_positions is None else None,
        showfliers=showfliers,
    )

    if title:
        ax.set_title(title)
    if xlabel:
        ax.set_xlabel(xlabel)
    if ylabel:
        ax.set_ylabel(ylabel)

    if tick_positions is not None:
        ax.set_xticks(tick_positions)
        ax.set_xticklabels([tick_labels[i - 1] for i in tick_positions])

    ax.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, bbox_inches="tight", dpi=300)

    plt.show()

In [ ]:
# Save path
save_path = None

# Data copy for boxplot purposes
boxplot_df = basis_df.copy()
boxplot_df["weekday_num"] = boxplot_df.index.dayofweek

# Year
boxplot_df["year"] = boxplot_df.index.year

# Hour of day
boxplot_df["hour"] = boxplot_df.index.hour
hourly_order = list(range(boxplot_df["hour"].min(), boxplot_df["hour"].max() + 1))

# Day of week
boxplot_df["weekday"] = boxplot_df.index.day_name()
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
boxplot_df["weekday"] = pd.Categorical(
    boxplot_df["weekday"], categories=weekday_order, ordered=True
)

# Week of year
boxplot_df["week_of_year"] = boxplot_df.index.isocalendar().week
weekly_order = sorted(boxplot_df["week_of_year"].unique())

# Month of year
boxplot_df["month"] = boxplot_df.index.month
boxplot_df["month_name"] = boxplot_df.index.strftime("%B")
monthly_order = [
    "January",
    "February",
    "March",
    "April",
    "May",
    "June",
    "July",
    "August",
    "September",
    "October",
    "November",
    "December",
]
boxplot_df["month_name"] = pd.Categorical(
    boxplot_df["month_name"], categories=monthly_order, ordered=True
)

### Basis

#### Basis vs Day of the Week

In [ ]:
title_base = f"Basis Distribution by Weekday\nFrom {basis_df.index.min().date()} To {basis_df.index.max().date()}"

for showfliers in [False, True]:
    save_path = (
        str(OUT_FIG_DIR / f"basis_by_weekday_{'with' if showfliers else 'without'}_outliers.png")
        if NOTEBOOK is not None
        else None
    )

    plot_grouped_boxplot(
        df=boxplot_df,
        value_col="Basis",
        group_col="weekday",
        order=weekday_order,
        tick_labels=weekday_order,
        showfliers=showfliers,
        title=f"{title_base}\nOutliers {'included' if showfliers else 'excluded'}",
        ylabel="Basis",
        xlabel="Day of Week",
        save_path=save_path,
    )

#### Basis Vs Hour of Day

In [ ]:
title_base = f"Basis Distribution by Hour of Day\nFrom {basis_df.index.min().date()} To {basis_df.index.max().date()}"

for showfliers in [False, True]:
    save_path = (
        str(OUT_FIG_DIR / f"basis_by_hour_{'with' if showfliers else 'without'}_outliers.png")
        if NOTEBOOK is not None
        else None
    )

    plot_grouped_boxplot(
        df=boxplot_df,
        value_col="Basis",
        group_col="hour",
        order=hourly_order,
        tick_labels=[f"{h:02d}:00" for h in hourly_order],
        showfliers=showfliers,
        title=f"{title_base}\nOutliers {'included' if showfliers else 'excluded'}",
        ylabel="Basis",
        xlabel="Hour of Day",
        save_path=save_path,
    )

#### Basis Vs Week of Year

In [ ]:
title_base = f"Basis Distribution by Week of Year\nFrom {basis_df.index.min().date()} To {basis_df.index.max().date()}"

tick_positions = [i for i, val in enumerate(weekly_order, start=1) if val % 4 == 0]
tick_labels = [f"W{val:02d}" for val in weekly_order]

for showfliers in [False, True]:
    save_path = (
        str(OUT_FIG_DIR / f"basis_by_week_{'with' if showfliers else 'without'}_outliers.png")
        if NOTEBOOK is not None
        else None
    )

    plot_grouped_boxplot(
        df=boxplot_df,
        value_col="Basis",
        group_col="week_of_year",
        order=weekly_order,
        tick_labels=tick_labels,
        tick_positions=tick_positions,
        showfliers=showfliers,
        title=f"{title_base}\nOutliers {'included' if showfliers else 'excluded'}",
        ylabel="Basis",
        xlabel="Week of Year",
        save_path=save_path,
    )

# Split by year
for year in boxplot_df["year"].unique():
    year_df = boxplot_df[boxplot_df["year"] == year]
    year_weekly_order = sorted(year_df["week_of_year"].unique())
    year_tick_positions = [i for i, val in enumerate(year_weekly_order, start=1) if val % 4 == 0]
    year_tick_labels = [f"W{val:02d}" for val in year_weekly_order]

    for showfliers in [False, True]:
        save_path = (
            str(
                OUT_FIG_DIR
                / f"basis_by_week_year_{year}_{'with' if showfliers else 'without'}_outliers.png"
            )
            if NOTEBOOK is not None
            else None
        )
        plot_grouped_boxplot(
            df=year_df,
            value_col="Basis",
            group_col="week_of_year",
            order=year_weekly_order,
            tick_labels=year_tick_labels,
            tick_positions=year_tick_positions,
            showfliers=showfliers,
            title=f"Basis Distribution by Week of Year ({year})\nOutliers {'included' if showfliers else 'excluded'}",
            ylabel="Basis",
            xlabel="Week of Year",
            save_path=save_path,
        )

#### Basis Vs Month of Year

In [ ]:
title_base = f"Basis Distribution by Month of Year\nFrom {basis_df.index.min().date()} To {basis_df.index.max().date()}"

for showfliers in [False, True]:
    save_path = (
        str(OUT_FIG_DIR / f"basis_by_month_{'with' if showfliers else 'without'}_outliers.png")
        if NOTEBOOK is not None
        else None
    )

    plot_grouped_boxplot(
        df=boxplot_df,
        value_col="Basis",
        group_col="month_name",
        order=monthly_order,
        tick_labels=monthly_order,
        showfliers=showfliers,
        title=f"{title_base}\nOutliers {'included' if showfliers else 'excluded'}",
        ylabel="Basis",
        xlabel="Month of Year",
        save_path=save_path,
    )

# Split by year
for year in boxplot_df["year"].unique():
    year_df = boxplot_df[boxplot_df["year"] == year]

    for showfliers in [False, True]:
        save_path = (
            str(
                OUT_FIG_DIR
                / f"basis_by_month_year_{year}_{'with' if showfliers else 'without'}_outliers.png"
            )
            if NOTEBOOK is not None
            else None
        )
        plot_grouped_boxplot(
            df=year_df,
            value_col="Basis",
            group_col="month_name",
            order=monthly_order,
            tick_labels=monthly_order,
            showfliers=showfliers,
            title=f"Basis Distribution by Month of Year ({year})\nOutliers {'included' if showfliers else 'excluded'}",
            ylabel="Basis",
            xlabel="Month of Year",
            save_path=save_path,
        )

### Basis 1st Difference

#### Basis 1st Difference Vs Hour of Day

In [ ]:
for key, freq in basis_diff_freqs.items():
    title_base = (
        f"Basis 1st Difference {key} Distribution by Dayhour\n"
        f"From {basis_df.index.min().date()} To {basis_df.index.max().date()}"
    )

    for showfliers in [False, True]:
        save_path = (
            str(
                OUT_FIG_DIR
                / f"basis_diff_{key}_by_hour_{'with' if showfliers else 'without'}_outliers.png"
            )
            if NOTEBOOK is not None
            else None
        )

        plot_grouped_boxplot(
            df=boxplot_df,
            value_col=f"Basis diff {key}",
            group_col="hour",
            order=hourly_order,
            tick_labels=[f"{h:02d}:00" for h in hourly_order],
            showfliers=showfliers,
            title=f"{title_base}\nOutliers {'included' if showfliers else 'excluded'}",
            ylabel="Basis 1st Difference",
            xlabel="Hour of Day",
            save_path=save_path,
        )

#### Basis 1st Difference Vs Day of Week

In [ ]:
for key, freq in basis_diff_freqs.items():
    title_base = (
        f"Basis 1st Difference {key} Distribution by Weekday\n"
        f"From {basis_df.index.min().date()} To {basis_df.index.max().date()}"
    )

    for showfliers in [False, True]:
        save_path = (
            str(
                OUT_FIG_DIR
                / f"basis_diff_{key}_by_weekday_{'with' if showfliers else 'without'}_outliers.png"
            )
            if NOTEBOOK is not None
            else None
        )

        plot_grouped_boxplot(
            df=boxplot_df,
            value_col=f"Basis diff {key}",
            group_col="weekday",
            order=weekday_order,
            tick_labels=weekday_order,
            showfliers=showfliers,
            title=f"{title_base}\nOutliers {'included' if showfliers else 'excluded'}",
            ylabel="Basis 1st Difference",
            xlabel="Day of Week",
            save_path=save_path,
        )

#### Basis 1st Difference Vs Week of Year

In [ ]:
for key, freq in basis_diff_freqs.items():
    title_base = (
        f"Basis 1st Difference {key} Distribution by Yearweek\n"
        f"From {basis_df.index.min().date()} To {basis_df.index.max().date()}"
    )

    tick_positions = [i for i, val in enumerate(weekly_order, start=1) if val % 4 == 0]
    tick_labels = [f"W{val:02d}" for val in weekly_order]

    for showfliers in [False, True]:
        save_path = (
            str(
                OUT_FIG_DIR
                / f"basis_diff_{key}_by_week_{'with' if showfliers else 'without'}_outliers.png"
            )
            if NOTEBOOK is not None
            else None
        )

        plot_grouped_boxplot(
            df=boxplot_df,
            value_col=f"Basis diff {key}",
            group_col="week_of_year",
            order=weekly_order,
            tick_labels=tick_labels,
            tick_positions=tick_positions,
            showfliers=showfliers,
            title=f"{title_base}\nOutliers {'included' if showfliers else 'excluded'}",
            ylabel="Basis Difference",
            xlabel="Week of Year",
            save_path=save_path,
        )

    # Split by year
    for year in boxplot_df["year"].unique():
        year_df = boxplot_df[boxplot_df["year"] == year]
        year_weekly_order = sorted(year_df["week_of_year"].unique())
        year_tick_positions = [
            i for i, val in enumerate(year_weekly_order, start=1) if val % 4 == 0
        ]
        year_tick_labels = [f"W{val:02d}" for val in year_weekly_order]

        for showfliers in [False, True]:
            save_path = (
                str(
                    OUT_FIG_DIR
                    / f"basis_diff_{key}_by_week_year_{year}_{'with' if showfliers else 'without'}_outliers.png"
                )
                if NOTEBOOK is not None
                else None
            )
            plot_grouped_boxplot(
                df=year_df,
                value_col=f"Basis diff {key}",
                group_col="week_of_year",
                order=year_weekly_order,
                tick_labels=year_tick_labels,
                tick_positions=year_tick_positions,
                showfliers=showfliers,
                title=f"Basis 1st Diff {key} by Week of Year ({year})\nOutliers {'included' if showfliers else 'excluded'}",
                ylabel="Basis Difference",
                xlabel="Week of Year",
                save_path=save_path,
            )

#### Basis 1st DIfference Vs Month of Year

In [ ]:
for key, freq in basis_diff_freqs.items():
    title_base = (
        f"Basis 1st Difference {key} Distribution by Yearmonth\n"
        f"From {basis_df.index.min().date()} To {basis_df.index.max().date()}"
    )

    for showfliers in [False, True]:
        save_path = (
            str(
                OUT_FIG_DIR
                / f"basis_diff_{key}_by_month_{'with' if showfliers else 'without'}_outliers.png"
            )
            if NOTEBOOK is not None
            else None
        )

        plot_grouped_boxplot(
            df=boxplot_df,
            value_col=f"Basis diff {key}",
            group_col="month_name",
            order=monthly_order,
            tick_labels=monthly_order,
            showfliers=showfliers,
            title=f"{title_base}\nOutliers {'included' if showfliers else 'excluded'}",
            ylabel="Basis 1st Difference",
            xlabel="Month of Year",
            save_path=save_path,
        )

    # Split by year
    for year in boxplot_df["year"].unique():
        year_df = boxplot_df[boxplot_df["year"] == year]

        for showfliers in [False, True]:
            save_path = (
                str(
                    OUT_FIG_DIR
                    / f"basis_diff_{key}_by_month_year_{year}_{'with' if showfliers else 'without'}_outliers.png"
                )
                if NOTEBOOK is not None
                else None
            )
            plot_grouped_boxplot(
                df=year_df,
                value_col=f"Basis diff {key}",
                group_col="month_name",
                order=monthly_order,
                tick_labels=monthly_order,
                showfliers=showfliers,
                title=f"Basis 1st Diff {key} by Month of Year ({year})\nOutliers {'included' if showfliers else 'excluded'}",
                ylabel="Basis 1st Difference",
                xlabel="Month of Year",
                save_path=save_path,
            )